# Task-Based Execution with the Python API

In [1]:
import openfe

**TODO: link to tutorial(s) on creating an alchemical network**

load an AlchemicalNetwork to execute - this is a small example campaign with only two edges and pre-charged ligands

In [2]:
alchemical_network = openfe.AlchemicalNetwork.from_json("alchemicalNetwork_mc1_small.json")

In [3]:
# build a database of tasks from the AlchemicalNetwork
from openfe.orchestration import setup_task_campaign

campaign_name = "mcl1"
warehouse_dir = f"warehouse_{campaign_name}"
db_path = f"tasks_{campaign_name}.db"

task_db, wh = setup_task_campaign(
    alchemical_network=alchemical_network,
    warehouse_dir=warehouse_dir,
    db_path=db_path
)

In [4]:
task_db

TaskStatusDB(engine=Engine(sqlite:///tasks_mcl1.db))

In [5]:
# verify that the alchemicalnetwork is stored
wh.exists(alchemical_network.key)

True

In [6]:
from openfe.orchestration import get_task_df, get_dependency_df

task_df = get_task_df(task_db);
task_df 

,taskid,status,last_modified,tries,max_tries
0,HybridTopologySetupUnit-8327cc87d598469b8766a6...,AVAILABLE,NaT,0,1
1,HybridTopologySetupUnit-8e920b2462ec4acba3e385...,AVAILABLE,NaT,0,1
2,HybridTopologySetupUnit-02f2da60546f40bdb07787...,AVAILABLE,NaT,0,1
3,HybridTopologySetupUnit-3024c161ec2246c2b5b6c6...,AVAILABLE,NaT,0,1
4,HybridTopologyMultiStateSimulationUnit-70fe09c...,BLOCKED,NaT,0,1
5,HybridTopologyMultiStateSimulationUnit-b3a19f6...,BLOCKED,NaT,0,1
6,HybridTopologyMultiStateSimulationUnit-d27bd1a...,BLOCKED,NaT,0,1
7,HybridTopologyMultiStateSimulationUnit-bfa5ad9...,BLOCKED,NaT,0,1
8,HybridTopologyMultiStateAnalysisUnit-7c00ce5c5...,BLOCKED,NaT,0,1
9,HybridTopologyMultiStateAnalysisUnit-675f13a80...,BLOCKED,NaT,0,1


In [7]:
dep_df = get_dependency_df(task_db);
dep_df

,from,to,blocking
0,HybridTopologySetupUnit-8327cc87d598469b8766a6...,HybridTopologyMultiStateSimulationUnit-70fe09c...,True
1,HybridTopologySetupUnit-8e920b2462ec4acba3e385...,HybridTopologyMultiStateSimulationUnit-b3a19f6...,True
2,HybridTopologySetupUnit-02f2da60546f40bdb07787...,HybridTopologyMultiStateSimulationUnit-d27bd1a...,True
3,HybridTopologySetupUnit-3024c161ec2246c2b5b6c6...,HybridTopologyMultiStateSimulationUnit-bfa5ad9...,True
4,HybridTopologySetupUnit-8327cc87d598469b8766a6...,HybridTopologyMultiStateAnalysisUnit-7c00ce5c5...,True
5,HybridTopologyMultiStateSimulationUnit-70fe09c...,HybridTopologyMultiStateAnalysisUnit-7c00ce5c5...,True
6,HybridTopologySetupUnit-8e920b2462ec4acba3e385...,HybridTopologyMultiStateAnalysisUnit-675f13a80...,True
7,HybridTopologyMultiStateSimulationUnit-b3a19f6...,HybridTopologyMultiStateAnalysisUnit-675f13a80...,True
8,HybridTopologySetupUnit-02f2da60546f40bdb07787...,HybridTopologyMultiStateAnalysisUnit-560fd27b0...,True
9,HybridTopologyMultiStateSimulationUnit-d27bd1a...,HybridTopologyMultiStateAnalysisUnit-560fd27b0...,True


## use a Worker to run the jobs

In [8]:
from openfe.orchestration import Worker
worker = Worker(wh, db_path)

In [9]:
# run this cell to execute units and do a status check
n_submissions = 4
for _ in range(n_submissions):
    exec_result = worker.execute_unit(scratch='scratch')
    print(exec_result)

INFO:openfe.utils.system_probe.log:SYSTEM CONFIG DETAILS:
INFO:openfe.utils.system_probe.log.hostname:hostname: 'Alyssas-Macbook-Pro.local'
INFO:openfe.utils.system_probe.log.gpu:CUDA-based GPU not found
INFO:openfe.utils.system_probe.log:Memory used: 20.0G (66.0%)
INFO:openfe.utils.system_probe.log:scratch/task_workdirs/HybridTopologySetupUnit-8327cc87d598469b8766a678206405fd: 73% full (248.0G free)
INFO:gufekey.openfe.protocols.openmm_rfe.hybridtop_units.HybridTopologySetupUnit:Starting system setup unit
INFO:gufekey.openfe.protocols.openmm_rfe.hybridtop_units.HybridTopologySetupUnit:Parameterizing systems
INFO:openmmforcefields.generators.template_generators:Requested to generate parameters for residue <Residue 153 (LIG) of chain 1>
INFO:openmmforcefields.generators.template_generators:Generating a residue template for [H][c]1[c]([H])[c]([H])[c]2[c]([c]1[H])[S][C]([C](=[O])[O-])=[C]2[Cl] using openff-2.2.1
INFO:openmmforcefields.generators.template_generators:Requested to generate p

('HybridTopologySetupUnit-8327cc87d598469b8766a678206405fd', ProtocolUnitResult(HybridTopology Setup: ligand_2 to ligand_3 repeat 0 generation 0))


INFO:openmmforcefields.generators.template_generators:Requested to generate parameters for residue <Residue 0 (LIG) of chain 0>
INFO:openmmforcefields.generators.template_generators:Generating a residue template for [H][c]1[c]([H])[c]([H])[c]2[c]([c]1[H])[S][C]([C](=[O])[O-])=[C]2[Cl] using openff-2.2.1
INFO:openmmforcefields.generators.template_generators:Requested to generate parameters for residue <Residue 594 (LIG) of chain 3>
INFO:openmmforcefields.generators.template_generators:Generating a residue template for [H][c]1[c]([H])[c]([Cl])[c]2[c]([c]1[H])[S][C]([C](=[O])[O-])=[C]2[Cl] using openff-2.2.1
INFO:openfe.protocols.openmm_rfe._rfe_utils.relative:Creating hybrid system
INFO:openfe.protocols.openmm_rfe._rfe_utils.relative:Setting force field terms
INFO:openfe.protocols.openmm_rfe._rfe_utils.relative:Adding forces
INFO:openfe.protocols.openmm_rfe._rfe_utils.relative:No CMAPTorsionForce found. Skipping adding force.
INFO:openfe.protocols.openmm_rfe._rfe_utils.relative:Hybrid sy

('HybridTopologySetupUnit-8e920b2462ec4acba3e3851d7763dfe6', ProtocolUnitResult(HybridTopology Setup: ligand_2 to ligand_3 repeat 0 generation 0))


INFO:openmmforcefields.generators.template_generators:Requested to generate parameters for residue <Residue 595 (LIG) of chain 3>
INFO:openmmforcefields.generators.template_generators:Generating a residue template for [H][c]1[c]([H])[c]([H])[c]2[c]([c]1[H])[S][C]([C](=[O])[O-])=[C]2[Cl] using openff-2.2.1
INFO:openfe.protocols.openmm_rfe._rfe_utils.relative:Creating hybrid system
INFO:openfe.protocols.openmm_rfe._rfe_utils.relative:Setting force field terms
INFO:openfe.protocols.openmm_rfe._rfe_utils.relative:Adding forces
INFO:openfe.protocols.openmm_rfe._rfe_utils.relative:No CMAPTorsionForce found. Skipping adding force.
INFO:openfe.protocols.openmm_rfe._rfe_utils.relative:Hybrid system created
/Users/atravitz/micromamba/envs/openfe_env/lib/python3.14/site-packages/mdtraj/core/topology.py:97: UserWarning: atom_indices are not monotonically increasing
  warnings.warn("atom_indices are not monotonically increasing")
INFO:exorcist.taskdb:Marking task 'HybridTopologySetupUnit-02f2da6054

('HybridTopologySetupUnit-02f2da60546f40bdb07787292f7345c3', ProtocolUnitResult(HybridTopology Setup: ligand_1 to ligand_2 repeat 0 generation 0))


INFO:openmmforcefields.generators.template_generators:Requested to generate parameters for residue <Residue 153 (LIG) of chain 1>
INFO:openmmforcefields.generators.template_generators:Generating a residue template for [H][C]1=[C]([C](=[O])[O-])[S][c]2[c]([H])[c]([H])[c]([H])[c]([H])[c]21 using openff-2.2.1
INFO:openmmforcefields.generators.template_generators:Requested to generate parameters for residue <Residue 8531 (LIG) of chain 4>
INFO:openmmforcefields.generators.template_generators:Generating a residue template for [H][c]1[c]([H])[c]([H])[c]2[c]([c]1[H])[S][C]([C](=[O])[O-])=[C]2[Cl] using openff-2.2.1
INFO:openfe.protocols.openmm_rfe._rfe_utils.relative:Creating hybrid system
INFO:openfe.protocols.openmm_rfe._rfe_utils.relative:Setting force field terms
INFO:openfe.protocols.openmm_rfe._rfe_utils.relative:Adding forces
INFO:openfe.protocols.openmm_rfe._rfe_utils.relative:No CMAPTorsionForce found. Skipping adding force.
INFO:openfe.protocols.openmm_rfe._rfe_utils.relative:Hybrid

('HybridTopologySetupUnit-3024c161ec2246c2b5b6c6eeed84fcbe', ProtocolUnitResult(HybridTopology Setup: ligand_1 to ligand_2 repeat 0 generation 0))


In [10]:
df=get_task_df(task_db)
df

,taskid,status,last_modified,tries,max_tries
0,HybridTopologySetupUnit-8327cc87d598469b8766a6...,COMPLETED,2026-09-29 08:20:03.881418,1,1
1,HybridTopologySetupUnit-8e920b2462ec4acba3e385...,COMPLETED,2026-09-29 08:20:04.585753,1,1
2,HybridTopologySetupUnit-02f2da60546f40bdb07787...,COMPLETED,2026-09-29 08:20:05.051724,1,1
3,HybridTopologySetupUnit-3024c161ec2246c2b5b6c6...,COMPLETED,2026-09-29 08:20:10.138807,1,1
4,HybridTopologyMultiStateSimulationUnit-70fe09c...,AVAILABLE,2026-09-29 08:20:03.881566,0,1
5,HybridTopologyMultiStateSimulationUnit-b3a19f6...,AVAILABLE,2026-09-29 08:20:04.585913,0,1
6,HybridTopologyMultiStateSimulationUnit-d27bd1a...,AVAILABLE,2026-09-29 08:20:05.051900,0,1
7,HybridTopologyMultiStateSimulationUnit-bfa5ad9...,AVAILABLE,2026-09-29 08:20:10.138942,0,1
8,HybridTopologyMultiStateAnalysisUnit-7c00ce5c5...,BLOCKED,NaT,0,1
9,HybridTopologyMultiStateAnalysisUnit-675f13a80...,BLOCKED,NaT,0,1


## Load results 
(cooking show magic)

**TODO: rerun to get results with 3 repeats per edge, upload to zenodo**

In [15]:
# load a different warehouse
from openfe.storage.warehouse import FileSystemWarehouse
# other_wh = FileSystemWarehouse.from_dir(warehouse_dir)

In [16]:
# result_edges = other_wh.gather_all_results();

In [19]:
# result_edges

In [18]:
# [(pr.get_estimate(), pr.get_uncertainty()) for pr, dag in result_edges if dag.ok()]